
**TPC-H Customer**

Dataset: samples.tpch.customer

- Difficulty: Easy
- Topics: filter, aggregation, groupBy

In [0]:
from pyspark.sql import functions as F, types as t

**Learn — TPC-H Basics: filter and groupBy**

In [0]:
# Run this example first — then solve the problems below.
# NOTE: this example is not a solution to any problem

df = spark.table("samples.tpch.customer")

# Explore schema and row count
print("Rows:", df.count())
df.printSchema()

# Group by nation_key (not market segment — the problems use mktsegment)
df.groupBy("c_nationkey").agg(
    F.count("*").alias("num_customers"),
    F.round(F.avg("c_acctbal"), 2).alias("avg_balance")
).orderBy(F.col("num_customers").desc()).show(5)

**Problem 1**

Count the number of customers in each market segment from the TPC-H customer table. Load samples.tpch.customer and sort by customer count descending.

Expected output columns:

- c_mktsegment - market segment name
- customer_count - number of customers in that segment (sorted descending)

In [0]:
result_1=df.groupBy("c_mktsegment").agg(F.count("c_custkey").alias("customer_count")).orderBy(F.col("customer_count").desc())
result_1.show()

In [0]:

# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you forget to assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'c_mktsegment' in cols, "Missing column: c_mktsegment"
assert 'customer_count' in cols, "Missing column: customer_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
counts = [r['customer_count'] for r in result_1.collect()]
assert counts == sorted(counts, reverse=True), "Results must be sorted by customer_count descending"
assert all(c > 0 for c in counts), "All customer counts must be positive"
print(f"Problem 1 passed ✓  ({cnt} rows)")

**Problem 2**

Find all customers who have a positive account balance (c_acctbal > 0). These are the customers with credit on their accounts.

Expected output columns:

- c_custkey - customer key
- c_name - customer name
- c_acctbal - account balance (must be > 0)

In [0]:
result_2=df.filter(F.col("c_acctbal")>0).select("c_custkey","c_name","c_acctbal")
result_2.show()


In [0]:

# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you forget to assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'c_custkey' in cols, "Missing column: c_custkey"
assert 'c_name' in cols, "Missing column: c_name"
assert 'c_acctbal' in cols, "Missing column: c_acctbal"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_bal = result_2.agg(F.min('c_acctbal')).collect()[0][0]
assert float(min_bal) > 0, f"All c_acctbal values must be > 0, found min={min_bal}"
print(f"Problem 2 passed ✓  ({cnt} rows)")
     

**Problem 3**

Calculate average, minimum, and maximum account balance broken down by market segment. This gives a financial profile for each customer segment.

Expected output columns:

- c_mktsegment - market segment
- avg_balance - average account balance for that segment
- min_balance - minimum account balance in that segment
- max_balance - maximum account balance in that segment

In [0]:
result_3=df.groupBy("c_mktsegment").agg(F.avg(F.col("c_acctbal")).alias("avg_balance"),F.max(F.col("c_acctbal")).alias("max_balance"),F.min(F.col("c_acctbal")).alias("min_balance"))
result_3.show()

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you forget to assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'c_mktsegment' in cols, "Missing column: c_mktsegment"
assert 'avg_balance' in cols, "Missing column: avg_balance"
assert 'min_balance' in cols, "Missing column: min_balance"
assert 'max_balance' in cols, "Missing column: max_balance"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
for row in result_3.collect():
    assert float(row['max_balance']) >= float(row['avg_balance']) >= float(row['min_balance']), \
        f"Expected min <= avg <= max for segment {row['c_mktsegment']}"
print(f"Problem 3 passed ✓  ({cnt} rows)")

**Problem 4**

Find all customers with a negative account balance (c_acctbal < 0). Sort the results ascending so the most negative balances appear first.

Expected output columns:

- c_custkey - customer key
- c_name - customer name
- c_acctbal - account balance (must be < 0), sorted ascending

In [0]:
result_4=df.filter(F.col("c_acctbal")<0).select("c_custkey","c_name","c_acctbal").orderBy(F.col("c_acctbal").asc())
result_4.show()
                                                                                                                                                                        

In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you forget to assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'c_custkey' in cols, "Missing column: c_custkey"
assert 'c_name' in cols, "Missing column: c_name"
assert 'c_acctbal' in cols, "Missing column: c_acctbal"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt >= 0, f"Expected rows >= 0, got {cnt}"
if cnt > 0:
    max_bal = result_4.agg(F.max('c_acctbal')).collect()[0][0]
    assert float(max_bal) < 0, f"All c_acctbal values must be < 0, found max={max_bal}"
    balances = [float(r['c_acctbal']) for r in result_4.collect()]
    assert balances == sorted(balances), "Results must be sorted by c_acctbal ascending"
print(f"Problem 4 passed ✓  ({cnt} rows)")

**Problem 5**

Count customers per nation key (c_nationkey) and return the top 10 nation keys by customer count. Nation keys link to the nation reference table.

Expected output columns:

- c_nationkey - nation key
- customer_count - number of customers mapped to that nation (top 10)

In [0]:
result_5=df.groupBy("c_nationkey").agg(F.count("c_custkey").alias("customer_count")).orderBy(F.col("customer_count").desc()).limit(10)
result_5.show()

In [0]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you forget to assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'c_nationkey' in cols, "Missing column: c_nationkey"
assert 'customer_count' in cols, "Missing column: customer_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt == 10, f"Expected exactly 10 rows (top 10), got {cnt}"
counts = [r['customer_count'] for r in result_5.collect()]
assert all(c > 0 for c in counts), "All customer_count values must be positive"
print(f"Problem 5 passed ✓  ({cnt} rows)")